# Cisco Skill Scanner — Dataset-Ready Automated Experiments

**MSR 2027 · SkillGuard · RQ1**

This notebook works with the **sample database or a full GitSkills SQLite database** using the same `artifacts` / `artifact_siblings` schema. It retrieves exact `SKILL.md` content, exports supporting files, performs four Cisco scans per skill, and produces comparison tables and CSV reports.

**Safety and scope:** Database access is read-only. Skill scripts are **never invoked directly by this notebook**; it launches the installed Cisco scanner against exported files. Treat downloaded skills as untrusted. Scan findings are not proof of malicious behavior. Full-folder completeness is limited to files actually captured in SQLite.

**No Terminal needed for normal scans.** Terminal may be needed to obtain the full database or install dependencies. This notebook never downloads the full dataset for you.

## 1 — Connect to your sample or full database

1. Open this notebook in `SkillGuard/notebooks` using the project `.venv` kernel.
2. Put the full SQLite database somewhere inside `SkillGuard/data/` **or** use an absolute path to an existing `.db`/`.sqlite` file.
3. Change `DATABASE` below. Use `"sample"` for your existing sample, or provide the full database's actual filename, for example `"data/full/<your-file>.db"` (replace the example!).
4. Run this cell. It prints discovered database candidates and validates the schema. A large database is queried incrementally, not loaded into memory.

**Note:** The full dataset must contain populated `artifacts` and `artifact_siblings` tables with compatible GitSkills columns. A different distribution (CSV/Parquet or different schema) needs an import/adapter first.

In [1]:
import csv
import hashlib
import json
import re
import shutil
import sqlite3
import subprocess
import tempfile
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

# EDIT THIS: "sample" or a real local path to the full SQLite file.
DATABASE = "sample"

cwd = Path.cwd().resolve()
ROOT = next((folder for folder in (cwd, *cwd.parents) if (folder / "notebooks").is_dir() and (folder / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the SkillGuard repository.")

candidates = sorted({*ROOT.glob("data/**/*.db"), *ROOT.glob("data/**/*.sqlite"), *ROOT.glob("data/**/*.sqlite3")})
print("Available local databases:")
for path in candidates[:30]:
    print(" -", path.relative_to(ROOT), f"({path.stat().st_size:,} bytes)")
if len(candidates) > 30:
    print(" ...", len(candidates)-30, "more")

if DATABASE == "sample":
    DB_PATH = ROOT / "data/sample/agent_skills_sample.db"
else:
    DB_PATH = Path(DATABASE).expanduser()
    if not DB_PATH.is_absolute():
        DB_PATH = ROOT / DB_PATH
DB_PATH = DB_PATH.resolve()
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Database not found: {DB_PATH}. Set DATABASE to the real path displayed above.")

# Never write to / modify the source dataset.
conn = sqlite3.connect(DB_PATH.as_uri() + "?mode=ro", uri=True)
conn.execute("PRAGMA query_only=ON")
conn.execute("PRAGMA busy_timeout=15000")

def sql(query, params=()):
    return pd.read_sql_query(query, conn, params=params)

def cols(table):
    return {row[1] for row in conn.execute(f"PRAGMA table_info({table})")}

required = {
    "artifacts": {"repo_full_name", "path", "content"},
    "artifact_siblings": {"repo_full_name", "artifact_path", "entry_name", "entry_type", "content", "content_fetched"},
}
for table, needed in required.items():
    missing = needed - cols(table)
    if missing:
        raise ValueError(f"Incompatible database schema: {table} lacks columns {sorted(missing)}")

DB_TAG = re.sub(r"[^a-zA-Z0-9_-]", "_", DB_PATH.stem)[:45]
print("\nConnected:", DB_PATH)
print("DB tag:", DB_TAG)
print("SQLite quick_check:", conn.execute("PRAGMA quick_check").fetchone()[0])
print("Artifact rows:", conn.execute("SELECT COUNT(*) FROM artifacts").fetchone()[0])

Available local databases:
 - data/sample/agent_skills_sample.db (290,955,264 bytes)

Connected: /Users/samantharies/PycharmProjects/MSR2027/SkillGuard/data/sample/agent_skills_sample.db
DB tag: agent_skills_sample
SQLite quick_check: ok
Artifact rows: 29786


## 2 — Select skills (single, search, or small batch)

- `MODE = "pilot"`: choose a known pilot by label.
- `MODE = "exact"`: specify an exact repository and `SKILL.md` path (works for any skill in the selected database).
- `MODE = "search"`: discover matches using a **case-insensitive keyword** in the `SKILL.md` body. Results are previewed only. Set `MODE="exact"` with the chosen repo/path to scan one, or choose `"batch"`.
- `MODE = "batch"`: select up to `MAX_SKILLS` matching records and scan them serially. Start small; scanning the entire corpus would take substantial time and disk space.

The keyword search may take time on a large SQLite database because it checks skill content. It only retrieves a limited number of result rows, not the entire table.

In [2]:
MODE = "pilot"  # "pilot", "exact", "search", or "batch"
PILOT = "canary-tokens"  # "loa-credentials", "canary-tokens", "x"
EXACT_REPO = "AceDataCloud/Skills"
EXACT_PATH = "skills/x/SKILL.md"
SEARCH_KEYWORD = "API_KEY"  # searched within SKILL.md text
PREVIEW_LIMIT = 15
MAX_SKILLS = 3  # batch safety cap; max 20 per run

PILOTS = {
    "loa-credentials": ("0xHoneyJar/loa-dixie", ".claude/skills/managing-credentials/SKILL.md"),
    "canary-tokens": ("26zl/cybersec-toolkit", ".claude/skills/implementing-canary-tokens-for-network-intrusion/SKILL.md"),
    "x": ("AceDataCloud/Skills", "skills/x/SKILL.md"),
}

assert MODE in ("pilot", "exact", "search", "batch")
assert 1 <= PREVIEW_LIMIT <= 100
assert 1 <= MAX_SKILLS <= 20, "Batch cap is 20 per Run All; split larger studies into batches."

def get_candidates(limit):
    # Always use SQL LIMIT rather than loading all rows.
    return sql("""SELECT repo_full_name, path,
                         substr(content, 1, 100) AS preview
                  FROM artifacts
                  WHERE content IS NOT NULL
                    AND instr(lower(content), lower(?)) > 0
                  ORDER BY repo_full_name, path LIMIT ?""", (SEARCH_KEYWORD, limit))

if MODE == "pilot":
    if PILOT not in PILOTS:
        raise ValueError(f"Unknown pilot: {PILOT}")
    selections = [PILOTS[PILOT]]
elif MODE == "exact":
    if not EXACT_REPO.strip() or not EXACT_PATH.strip():
        raise ValueError("Enter exact repository and SKILL.md path")
    selections = [(EXACT_REPO, EXACT_PATH)]
elif MODE == "search":
    matches = get_candidates(PREVIEW_LIMIT)
    print(f"Previewing up to {PREVIEW_LIMIT} matches for {SEARCH_KEYWORD!r}:")
    display(matches)
    selections = []  # preview only; NO scans
else:
    matches = get_candidates(MAX_SKILLS)
    display(matches)
    selections = list(matches[["repo_full_name", "path"]].itertuples(index=False, name=None))

print(f"Selected {len(selections)} skill(s).")
for repo, path in selections:
    print(" -", repo, "|", path)
if MODE == "search":
    print("SEARCH MODE: preview only. Switch to exact or batch to export/scan.")

Selected 1 skill(s).
 - 26zl/cybersec-toolkit | .claude/skills/implementing-canary-tokens-for-network-intrusion/SKILL.md


## 3 — Configure and run the experiment

**Single click:** set `RUN_SCANS = True` and run this cell after Sections 1–2. The code will: retrieve the selected skill(s), export a `SKILL.md`-only and full-folder version, verify content, run Cisco default + behavioral against each, save validated JSON, and build a summary CSV. **No Terminal commands are required.**

**Controls:** `RUN_SCANS=False` exports and attempts to read previously saved results without running Cisco. `ALLOW_INCOMPLETE_FOLDERS=False` prevents scanning a supposedly full folder when GitSkills did not capture every supporting file. If you deliberately scan an incomplete folder, report that limitation.

For safety/reproducibility, output paths include database name plus a unique hash of repo/path. The notebook never executes the skill's own scripts. Cisco analyzer behavior is determined by the installed scanner; do not treat it as a sandbox.

In [3]:
RUN_SCANS = True
ALLOW_INCOMPLETE_FOLDERS = False
SCAN_TIMEOUT_SECONDS = 300

scanner_options = [ROOT / ".venv/bin/skill-scanner", ROOT / ".venv/Scripts/skill-scanner.exe"]
SCANNER = next((str(path) for path in scanner_options if path.is_file()), None) or shutil.which("skill-scanner")
if RUN_SCANS and selections and not SCANNER:
    raise FileNotFoundError("Cisco skill-scanner not found. Select the SkillGuard .venv kernel or install the scanner.")

run_root = ROOT / "results" / "cisco_dataset_runs" / DB_TAG
run_root.mkdir(parents=True, exist_ok=True)


def key_for_skill(repo, skill_path):
    stem = re.sub(r"[^a-zA-Z0-9_-]", "-", Path(skill_path).parent.name)[:35].strip("-") or "skill"
    digest = hashlib.sha256(f"{repo}\n{skill_path}".encode()).hexdigest()[:12]
    return f"{stem}-{digest}"


def secure_target(folder, entry):
    # GitSkills sibling paths should be skill-folder-relative, e.g. scripts/x.py.
    rel = Path(entry)
    if (not entry or rel.is_absolute() or ".." in rel.parts or "\\" in entry or rel.name == "SKILL.md"):
        raise ValueError(f"Unsafe or conflicting supporting file path: {entry!r}")
    target = folder / rel
    if not target.resolve().is_relative_to(folder.resolve()):
        raise ValueError(f"Path escapes folder: {entry!r}")
    return target


def write_exact(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_text(encoding="utf-8") != content:
            raise ValueError(f"File differs from prior export; refusing overwrite: {path}")
    else:
        path.write_text(content, encoding="utf-8")


def read_report(path):
    if not path.is_file():
        return None, "MISSING"
    raw = path.read_text(encoding="utf-8", errors="replace")
    if not raw.strip():
        return None, "EMPTY"
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        return None, "INVALID JSON"
    if not isinstance(obj, dict) or not isinstance(obj.get("findings"), list) or not isinstance(obj.get("is_safe"), bool):
        return None, "INVALID FORMAT"
    return obj, "VALID"


def run_scanner(folder, filename):
    args = [SCANNER, "scan", str(folder), "--format", "json"]
    if "behavioral" in filename:
        args.insert(3, "--use-behavioral")
    try:
        proc = subprocess.run(args, cwd=ROOT, capture_output=True, text=True,
                              timeout=SCAN_TIMEOUT_SECONDS, check=False)
    except subprocess.TimeoutExpired:
        return None, "TIMEOUT"
    try:
        obj = json.loads(proc.stdout)
        if not isinstance(obj, dict) or not isinstance(obj.get("findings"), list) or not isinstance(obj.get("is_safe"), bool):
            raise ValueError("Unexpected JSON format")
    except (json.JSONDecodeError, ValueError):
        print("Scanner error:", (proc.stderr or proc.stdout)[-400:])
        return None, f"SCANNER ERROR (exit={proc.returncode})"
    return obj, "VALID" if proc.returncode == 0 else f"VALID JSON, exit={proc.returncode}"

scan_configs = [
    ("SKILL.md only | default", "only", "skill_only_default.json"),
    ("SKILL.md only | behavioral", "only", "skill_only_behavioral.json"),
    ("Full folder | default", "full", "full_folder_default.json"),
    ("Full folder | behavioral", "full", "full_folder_behavioral.json"),
]
summary = []
findings = []
inventory = []

if RUN_SCANS and selections:
    ver = subprocess.run([SCANNER, "--version"], capture_output=True, text=True, timeout=30)
    print("Cisco scanner:", (ver.stdout or ver.stderr).strip())

for number, (repo, skill_path) in enumerate(selections, 1):
    skill_id = key_for_skill(repo, skill_path)
    skill_dir = run_root / skill_id
    folder_only = skill_dir / "only"
    folder_full = skill_dir / "full"
    reports = skill_dir / "reports"
    reports.mkdir(parents=True, exist_ok=True)
    print(f"\n[{number}/{len(selections)}] {repo}/{skill_path}")
    try:
        match = sql("SELECT content FROM artifacts WHERE repo_full_name=? AND path=? AND content IS NOT NULL LIMIT 1", (repo, skill_path))
        if match.empty:
            raise ValueError("SKILL.md content unavailable in selected DB")
        skill_text = match.iloc[0]["content"]
        if not isinstance(skill_text, str) or not skill_text.strip():
            raise ValueError("SKILL.md empty")
        files = sql("""SELECT entry_name, entry_type, content, content_fetched
                       FROM artifact_siblings
                       WHERE repo_full_name=? AND artifact_path=?
                       ORDER BY entry_name""", (repo, skill_path))
        metadata_cols = cols("artifacts")
        fields = [f for f in ("composition_fetched", "composition_truncated", "sibling_count") if f in metadata_cols]
        meta = sql(f'SELECT {", ".join(fields)} FROM artifacts WHERE repo_full_name=? AND path=? LIMIT 1', (repo, skill_path)).iloc[0] if fields else None
        write_exact(folder_only / "SKILL.md", skill_text)
        write_exact(folder_full / "SKILL.md", skill_text)
        missing = []
        for item in files.itertuples(index=False):
            if item.entry_type != "file":
                continue
            target = secure_target(folder_full, item.entry_name)
            if item.content_fetched != 1 or not isinstance(item.content, str):
                missing.append(item.entry_name)
                continue
            write_exact(target, item.content)
        if (folder_only / "SKILL.md").read_bytes() != (folder_full / "SKILL.md").read_bytes():
            raise AssertionError("SKILL.md mismatch")
        composition_ok = bool(meta is not None and "composition_fetched" in fields and meta["composition_fetched"] == 1 and
                              "composition_truncated" in fields and meta["composition_truncated"] == 0)
        complete = composition_ok and not missing
        inv = {"database": str(DB_PATH), "repo": repo, "skill_path": skill_path,
               "skill_id": skill_id, "composition_complete": complete,
               "missing_files": missing,
               "files": [{"path": str(p.relative_to(folder_full)), "sha256": hashlib.sha256(p.read_bytes()).hexdigest()}
                         for p in sorted(folder_full.rglob("*")) if p.is_file()]}
        (skill_dir / "manifest.json").write_text(json.dumps(inv, indent=2) + "\n", encoding="utf-8")
        print("  Exported", len(inv["files"]), "file(s). Full-folder complete:", complete)
        if missing:
            print("  Missing content:", missing[:10])
        inventory.append(inv)
        for label, kind, filename in scan_configs:
            row = {"database": DB_TAG, "repo": repo, "skill_path": skill_path, "skill_id": skill_id,
                   "configuration": label, "composition_complete": complete,
                   "status": "", "findings_count": None, "max_severity": None, "is_safe": None}
            report_file = reports / filename
            if RUN_SCANS:
                if kind == "full" and not complete and not ALLOW_INCOMPLETE_FOLDERS:
                    row["status"] = "SKIPPED: INCOMPLETE FOLDER"
                    obj = None
                else:
                    obj, row["status"] = run_scanner(folder_only if kind == "only" else folder_full, filename)
                    if obj is not None:
                        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=reports,
                                                         prefix=".tmp_cisco_", suffix=".json", delete=False) as stream:
                            tmp = Path(stream.name)
                            json.dump(obj, stream, indent=2)
                            stream.write("\n")
                        tmp.replace(report_file)
            else:
                obj, row["status"] = read_report(report_file)
            if obj is not None:
                row.update(findings_count=len(obj["findings"]), max_severity=obj.get("max_severity"), is_safe=obj["is_safe"])
                for f in obj["findings"]:
                    if not isinstance(f, dict):
                        continue
                    findings.append({"repo": repo, "skill_path": skill_path, "skill_id": skill_id,
                                     "configuration": label, "severity": f.get("severity"),
                                     "rule_id": f.get("rule_id"), "analyzer": f.get("analyzer"),
                                     "file_path": f.get("file_path"), "line_number": f.get("line_number"),
                                     "title": f.get("title"), "description": f.get("description")})
            summary.append(row)
            print(" ", label, "->", row["status"], "/", row["findings_count"], "findings")
    except Exception as exc:
        # One unavailable artifact must not stop the entire batch.
        print("  ERROR:", type(exc).__name__, str(exc)[:250])
        summary.append({"database": DB_TAG, "repo": repo, "skill_path": skill_path, "skill_id": skill_id,
                        "configuration": "export", "status": f"ERROR: {type(exc).__name__}: {exc}",
                        "composition_complete": None, "findings_count": None,
                        "max_severity": None, "is_safe": None})

summary_df = pd.DataFrame(summary)
findings_df = pd.DataFrame(findings)
summary_df.to_csv(run_root / "scan_summary.csv", index=False)
findings_df.to_csv(run_root / "individual_findings.csv", index=False)
print("\nSaved results:", run_root)
print("Note: missing/incomplete/error runs are NOT zero findings.")

Cisco scanner: skill-scanner 2.3.1

[1/1] 26zl/cybersec-toolkit/.claude/skills/implementing-canary-tokens-for-network-intrusion/SKILL.md
  Exported 4 file(s). Full-folder complete: True
  SKILL.md only | default -> VALID / 0 findings
  SKILL.md only | behavioral -> VALID / 4 findings
  Full folder | default -> VALID / 12 findings
  Full folder | behavioral -> VALID / 22 findings

Saved results: /Users/samantharies/PycharmProjects/MSR2027/SkillGuard/results/cisco_dataset_runs/agent_skills_sample
Note: missing/incomplete/error runs are NOT zero findings.


## 4 — Review results and compare contexts

A row labeled `SKIPPED`, `MISSING`, or `ERROR` is **not** a clean scan. Review false positives and compare `SKILL.md` alone with its full folder. The generated `scan_summary.csv` and `individual_findings.csv` can be used in your research notes.

In [4]:
print("SCAN SUMMARY")
if summary_df.empty:
    print("No skills selected. Choose pilot, exact, or batch in Section 2.")
else:
    display(summary_df)
print("\nINDIVIDUAL FINDINGS")
if findings_df.empty:
    print("No findings in available reports (or reports were skipped / missing). Check statuses above.")
else:
    display(findings_df)

if not summary_df.empty and "configuration" in summary_df:
    good = summary_df[summary_df["status"].astype(str).str.startswith("VALID")]
    if not good.empty:
        print("\nFINDING-COUNT COMPARISON")
        display(good.pivot_table(index=["repo", "skill_path"], columns="configuration",
                                 values="findings_count", aggfunc="first"))

print("\nFiles saved under:", run_root)

SCAN SUMMARY


,database,repo,skill_path,skill_id,configuration,composition_complete,status,findings_count,max_severity,is_safe
0,agent_skills_sample,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,SKILL.md only | default,True,VALID,0,SAFE,True
1,agent_skills_sample,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,SKILL.md only | behavioral,True,VALID,4,MEDIUM,True
2,agent_skills_sample,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,Full folder | default,True,VALID,12,CRITICAL,False
3,agent_skills_sample,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,Full folder | behavioral,True,VALID,22,CRITICAL,False



INDIVIDUAL FINDINGS


,repo,skill_path,skill_id,configuration,severity,rule_id,analyzer,file_path,line_number,title,description
0,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,SKILL.md only | behavioral,MEDIUM,MDBLOCK_PYTHON_HTTP_POST,behavioral,SKILL.md,65.0,Python code block sends HTTP POST request,Code block in SKILL.md at line 65 contains pot...
1,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,SKILL.md only | behavioral,MEDIUM,MDBLOCK_PYTHON_HTTP_POST,behavioral,SKILL.md,91.0,Python code block sends HTTP POST request,Code block in SKILL.md at line 91 contains pot...
2,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,SKILL.md only | behavioral,MEDIUM,MDBLOCK_PYTHON_HTTP_POST,behavioral,SKILL.md,117.0,Python code block sends HTTP POST request,Code block in SKILL.md at line 117 contains po...
3,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,SKILL.md only | behavioral,MEDIUM,MDBLOCK_PYTHON_HTTP_POST,behavioral,SKILL.md,165.0,Python code block sends HTTP POST request,Code block in SKILL.md at line 165 contains po...
4,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,Full folder | default,MEDIUM,TOOL_ABUSE_UNDECLARED_NETWORK,static,SKILL.md,NaN,Undeclared network usage,Skill code uses network libraries but doesn't ...
5,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,Full folder | default,HIGH,CORRELATED_SENSITIVE_NETWORK_FLOW,correlation,scripts/agent.py,91.0,Sensitive source reaches a network sink,Correlated sensitive_environment → network beh...
6,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,Full folder | default,MEDIUM,DATA_EXFIL_NETWORK_REQUESTS,static,scripts/agent.py,91.0,Outbound network request primitives that can t...,Pattern detected: requests.post(
7,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,Full folder | default,MEDIUM,DATA_EXFIL_NETWORK_REQUESTS,static,scripts/agent.py,179.0,Outbound network request primitives that can t...,Pattern detected: requests.post(
8,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,Full folder | default,MEDIUM,DATA_EXFIL_NETWORK_REQUESTS,static,scripts/agent.py,187.0,Outbound network request primitives that can t...,Pattern detected: requests.get(
9,26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-...,implementing-canary-tokens-for-netw-460e0a897b56,Full folder | default,MEDIUM,DATA_EXFIL_NETWORK_REQUESTS,static,scripts/agent.py,195.0,Outbound network request primitives that can t...,Pattern detected: requests.get(



FINDING-COUNT COMPARISON


,configuration,Full folder | behavioral,Full folder | default,SKILL.md only | behavioral,SKILL.md only | default
repo,skill_path,,,,
26zl/cybersec-toolkit,.claude/skills/implementing-canary-tokens-for-network-intrusion/SKILL.md,22,12,4,0



Files saved under: /Users/samantharies/PycharmProjects/MSR2027/SkillGuard/results/cisco_dataset_runs/agent_skills_sample


## 5 — Research interpretation & full-dataset limitations

- **Controlled comparison:** Each SKILL.md-only scan and full-folder scan uses identical `SKILL.md` bytes. More context is the only intended input difference.
- **Scanner scope:** A reported network or credential behavior may be legitimate. `is_safe=True` can coexist with findings; do not equate it with an absence of risks.
- **Incomplete capture:** Some GitSkills sibling files were not fetched or are binary. Missing files and truncated compositions must be excluded from clean full-folder comparisons (or reported separately).
- **Scale:** The batch selector purposely caps runs at 20 selected skills per execution; extend via multiple documented batches, not a massive Run All. Each skill uses 4 Cisco invocations. Keyword search may be slow on huge databases; for systematic sampling, select with appropriate SQL filters and preserve the sampling protocol.
- **Reproducibility:** Record DB version/checksum, scanner version, skill identifiers, completeness and exported file hashes. The notebook writes `manifest.json` per skill plus CSV reports.
- **Terminal only if needed:** To obtain or install the full SQLite dataset and scanner, use normal project setup outside Jupyter. Scanning itself happens inside Jupyter. Avoid committing large databases or exported untrusted skills to GitHub without team agreement.